#FAKE NEWS DETECTION SYSTEM

## ==========================================
## PIPELINE 1: DATA COLLECTION
## ==========================================

In [6]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset

print("=" * 60)
print("PIPELINE 1: DATA COLLECTION")
print("=" * 60)

# Download the WELFake dataset directly from Hugging Face repositories
print("Downloading davanstrien/WELFake dataset from Hugging Face...")
dataset = load_dataset("davanstrien/WELFake")

# Print collection summary and split statistics
print("\n--- Collection Summary ---")
print(f"Dataset Splits Available: {list(dataset.keys())}")
for split_name, split_data in dataset.items():
    print(f" -> Split '{split_name}': {len(split_data):,} rows")
print(f"Dataset Columns: {dataset['train'].column_names}")
print("=" * 60 + "\n")

PIPELINE 1: DATA COLLECTION

--- Collection Summary ---
Dataset Splits Available: ['train']
 -> Split 'train': 72,134 rows
Dataset Columns: ['title', 'text', 'label']



## ==========================================
## PIPELINE 2: DATA PREPARATION
## ==========================================

In [9]:
# ==========================================
# PIPELINE 2: DATA PREPARATION
# ==========================================
print("=" * 60)
print("PIPELINE 2: DATA PREPARATION")
print("=" * 60)

initial_train_size = len(dataset['train'])

# Clean missing values and format features safely using dictionary batch parsing
def clean_and_format(batch):
    # Extract columns safely from the batch dictionary
    titles = batch.get('title', [''] * len(batch['label']))
    texts = batch.get('text', [''] * len(batch['label']))
    labels = batch['label']

    combined_texts = []
    cleaned_labels = []

    for t, txt, lbl in zip(titles, texts, labels):
        # Handle None types or missing values safely
        t_str = str(t) if t is not None else ""
        txt_str = str(txt) if txt is not None else ""

        # Combine title and text
        combined = (t_str + " " + txt_str).strip()

        # Filter out extremely short or invalid rows (> 20 characters)
        if len(combined) > 20:
            combined_texts.append(combined)
            cleaned_labels.append(lbl)

    return {
        'text': combined_texts,
        'label': cleaned_labels
    }

print("Cleaning missing values and merging 'title' + 'text' features safely...")
prepared_dataset = dataset.map(
    clean_and_format,
    batched=True,
    remove_columns=dataset['train'].column_names
)

final_train_size = len(prepared_dataset['train'])
print("\n--- Preparation Summary ---")
print(f"Initial Training Rows: {initial_train_size:,}")
print(f"Cleaned Training Rows: {final_train_size:,} (Dropped {initial_train_size - final_train_size} invalid/short records)")
print(f"Sample Prepared Text Preview:\n\"{prepared_dataset['train']['text'][0][:180]}...\"")
print(f"Sample Corresponding Label: {prepared_dataset['train']['label'][0]} (0: Fake, 1: Real)")
print("=" * 60 + "\n")

PIPELINE 2: DATA PREPARATION
Cleaning missing values and merging 'title' + 'text' features safely...


Map:   0%|          | 0/72134 [00:00<?, ? examples/s]


--- Preparation Summary ---
Initial Training Rows: 72,134
Cleaned Training Rows: 72,085 (Dropped 49 invalid/short records)
Sample Prepared Text Preview:
"LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members ..."
Sample Corresponding Label: 1 (0: Fake, 1: Real)



## ==========================================
## PIPELINE 3: DATA PREPROCESSING (TOKENIZATION)
## ==========================================

In [10]:
from transformers import AutoTokenizer

print("=" * 60)
print("PIPELINE 3: DATA PREPROCESSING (TOKENIZATION)")
print("=" * 60)

model_name = "distilbert-base-uncased"
max_length = 512

print(f"Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization mapping function with padding and truncation limits
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

print(f"Tokenizing text sequences (Max Length: {max_length})...")
tokenized_dataset = prepared_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("\n--- Preprocessing Summary ---")
print(f"Tokenizer Vocabulary Size: {tokenizer.vocab_size:,} tokens")
print(f"Tokenized Train Dataset Features: {tokenized_dataset['train'].column_names}")
print(f"Sample Input IDs Shape: {np.array(tokenized_dataset['train'][0]['input_ids']).shape}")
print("=" * 60 + "\n")

PIPELINE 3: DATA PREPROCESSING (TOKENIZATION)
Loading tokenizer: distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing text sequences (Max Length: 512)...


Map:   0%|          | 0/72085 [00:00<?, ? examples/s]


--- Preprocessing Summary ---
Tokenizer Vocabulary Size: 30,522 tokens
Tokenized Train Dataset Features: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Sample Input IDs Shape: (512,)



## ==========================================
## PIPELINE 4: DATA BALANCING & SAMPLING
## ==========================================

In [13]:
# ==========================================
# PIPELINE 4: DATA BALANCING & TRAIN-TEST SPLIT
# ==========================================
from datasets import Dataset, DatasetDict
import pandas as pd
import numpy as np

print("=" * 60)
print("PIPELINE 4: DATA BALANCING & TRAIN-TEST SPLIT")
print("=" * 60)

# 1. Convert tokenized dataset back to a pandas DataFrame for easy stratified balancing
df_train = pd.DataFrame({
    'input_ids': tokenized_dataset['train']['input_ids'],
    'attention_mask': tokenized_dataset['train']['attention_mask'],
    'label': tokenized_dataset['train']['label']
})

# 2. Separate into Fake (0) and Real (1) classes
df_fake = df_train[df_train['label'] == 0]
df_real = df_train[df_train['label'] == 1]

print(f"Total available Fake news rows: {len(df_fake):,}")
print(f"Total available Real news rows: {len(df_real):,}")

# 3. Define target sample size per class to fit safely in Colab T4 GPU memory
# (e.g., 5,000 of each = 10,000 total rows trains in ~8 mins with great accuracy)
samples_per_class = 5000

df_fake_sampled = df_fake.sample(n=min(samples_per_class, len(df_fake)), random_state=42)
df_real_sampled = df_real.sample(n=min(samples_per_class, len(df_real)), random_state=42)

# 4. Combine and shuffle balanced data
df_balanced = pd.concat([df_fake_sampled, df_real_sampled]).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"\nBalanced Dataset Created:")
print(f" -> Fake News (0): {len(df_fake_sampled):,}")
print(f" -> Real News (1): {len(df_real_sampled):,}")
print(f" -> Total Combined Rows: {len(df_balanced):,}")

# 5. Perform explicit Train-Test Split (80% Train, 20% Test)
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_balanced,
    test_size=0.20,
    stratify=df_balanced['label'],
    random_state=42
)

# 6. Convert pandas DataFrames back into Hugging Face Dataset format
train_ds = Dataset.from_pandas(train_df.drop(columns=['__index_level_0__'], errors='ignore'))
test_ds = Dataset.from_pandas(test_df.drop(columns=['__index_level_0__'], errors='ignore'))

tokenized_dataset = DatasetDict({
    'train': train_ds,
    'test': test_ds
})

print("\n--- Split & Balancing Summary ---")
print(f"Training Split Size: {len(tokenized_dataset['train']):,} samples")
print(f"Testing Split Size: {len(tokenized_dataset['test']):,} samples")
print("Dataset successfully balanced and split, ready for T4 GPU training!")
print("=" * 60 + "\n")

PIPELINE 4: DATA BALANCING & TRAIN-TEST SPLIT
Total available Fake news rows: 7,055
Total available Real news rows: 7,362

Balanced Dataset Created:
 -> Fake News (0): 5,000
 -> Real News (1): 5,000
 -> Total Combined Rows: 10,000

--- Split & Balancing Summary ---
Training Split Size: 8,000 samples
Testing Split Size: 2,000 samples
Dataset successfully balanced and split, ready for T4 GPU training!



## ==========================================
## PIPELINE 5: MODEL BUILDING
## ==========================================

In [14]:
from transformers import AutoModelForSequenceClassification

print("=" * 60)
print("PIPELINE 5: MODEL BUILDING")
print("=" * 60)

num_labels = 2
print(f"Building sequence classification model from base: {model_name}")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n--- Model Architecture Summary ---")
print(f"Base Architecture: DistilBERT For Sequence Classification")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Target Output Classes: {num_labels} (Binary)")
print("=" * 60 + "\n")

PIPELINE 5: MODEL BUILDING
Building sequence classification model from base: distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Model Architecture Summary ---
Base Architecture: DistilBERT For Sequence Classification
Total Parameters: 66,955,010
Trainable Parameters: 66,955,010
Target Output Classes: 2 (Binary)



## ==========================================
## PIPELINE 6: MODEL TRAINING
## ==========================================

In [16]:
# ==========================================
# PIPELINE 6: MODEL TRAINING
# ==========================================
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("=" * 60)
print("PIPELINE 6: MODEL TRAINING")
print("=" * 60)

output_dir = "./results"
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True, # Enables T4 GPU mixed-precision acceleration
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

# Compute performance metrics during evaluation epochs
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

# Initialized Trainer without the deprecated 'tokenizer' argument (since data is already pre-tokenized)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset.get('test', tokenized_dataset.get('validation', None)),
    compute_metrics=compute_metrics,
)

print("Launching fine-tuning on free-tier T4 GPU...")
training_output = trainer.train()

print("\n--- Training Completion Summary ---")
print(f"Training Loss Output: {training_output.metrics}")
print("=" * 60 + "\n")

PIPELINE 6: MODEL TRAINING
Launching fine-tuning on free-tier T4 GPU...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.069576,0.073626,0.980000,0.979920,0.983871,0.976000
2,0.043695,0.084013,0.979000,0.978937,0.981891,0.976000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Training Completion Summary ---
Training Loss Output: {'train_runtime': 242.9199, 'train_samples_per_second': 65.865, 'train_steps_per_second': 4.117, 'total_flos': 2119478378496000.0, 'train_loss': 0.08090367642045021, 'epoch': 2.0}



## ==========================================
## PIPELINE 7: MODEL TESTING
## ==========================================

In [17]:
print("=" * 60)
print("PIPELINE 7: MODEL TESTING")
print("=" * 60)

test_split = tokenized_dataset.get('test', tokenized_dataset.get('validation'))
print("Executing evaluation sweep on test split...")
eval_results = trainer.evaluate(test_split)

print("\n--- Test Results Metric Breakdown ---")
for metric_name, value in eval_results.items():
    print(f" -> {metric_name}: {value:.4f}" if isinstance(value, float) else f" -> {metric_name}: {value}")
print("=" * 60 + "\n")

PIPELINE 7: MODEL TESTING
Executing evaluation sweep on test split...


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.043695,0.073626,2,0.980000,0.979920,0.983871,0.976000



--- Test Results Metric Breakdown ---
 -> eval_loss: 0.0736
 -> eval_accuracy: 0.9800
 -> eval_f1: 0.9799
 -> eval_precision: 0.9839
 -> eval_recall: 0.9760



## ==========================================
## PIPELINE 8: MODEL EVALUATION & GENERALIZATION REPORT
## ==========================================

In [22]:

from sklearn.metrics import classification_report
import torch
import torch.nn.functional as F

print("=" * 60)
print("PIPELINE 8: MODEL EVALUATION & GENERALIZATION REPORT")
print("=" * 60)

# 1. Standard Test Split Evaluation & Classification Report
predictions_output = trainer.predict(test_split)
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

print("\n--- Detailed Quantitative Classification Report ---")
report = classification_report(labels, preds, target_names=["Fake News (0)", "Real News (1)"])
print(report)

# 2. Generalization Report with Explicit Real-World Examples
print("\n" + "=" * 10 + " GENERALIZATION REPORT (EXPLICIT SAMPLES) " + "=" * 10)

# Curated test examples representing distinct types of articles
explicit_test_samples = [
    {
        "text": "BREAKING: NASA confirms that a newly discovered asteroid is made entirely of solid gold and will land in the Atlantic Ocean tomorrow.",
        "expected": "Fake News (0)"
    },
    {
        "text": "Federal Reserve officials announced another quarter-point interest rate cut following cooler inflation reports released on Tuesday morning.",
        "expected": "Real News (1)"
    },
    {
        "text": "SHOCKING SECRET: Local doctor discovers one weird trick to completely reverse aging in 3 days using kitchen ingredients!",
        "expected": "Fake News (0)"
    },
    {
        "text": "The international science team published new measurements of Antarctic ice sheet velocities in the journal Nature Geoscience.",
        "expected": "Real News (1)"
    }
]

# Set model to evaluation mode for inference
model.eval()
label_map = { 0: "Real News ✅",1: "Fake News 🛑",}

for idx, sample in enumerate(explicit_test_samples, 1):
    # Tokenize the custom sample text
    inputs = tokenizer(
        sample["text"],
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    # Move inputs to the same device as the model (GPU/CPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        pred_label_id = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_label_id].item() * 100

    predicted_text = label_map[pred_label_id]

    print(f"\n[Example {idx}]")
    print(f"Snippet   : \"{sample['text'][:100]}...\"")
    print(f"Expected  : {sample['expected']}")
    print(f"Predicted : {predicted_text}")
    print(f"Confidence: {confidence:.2f}%")
    print("-" * 55)

print("=" * 60 + "\n")

PIPELINE 8: MODEL EVALUATION & GENERALIZATION REPORT



--- Detailed Quantitative Classification Report ---
               precision    recall  f1-score   support

Fake News (0)       0.98      0.98      0.98      1000
Real News (1)       0.98      0.98      0.98      1000

     accuracy                           0.98      2000
    macro avg       0.98      0.98      0.98      2000
 weighted avg       0.98      0.98      0.98      2000


========== GENERALIZATION REPORT (EXPLICIT SAMPLES) ==========

[Example 1]
Snippet   : "BREAKING: NASA confirms that a newly discovered asteroid is made entirely of solid gold and will lan..."
Expected  : Fake News (0)
Predicted : Fake News 🛑
Confidence: 99.41%
-------------------------------------------------------

[Example 2]
Snippet   : "Federal Reserve officials announced another quarter-point interest rate cut following cooler inflati..."
Expected  : Real News (1)
Predicted : Real News ✅
Confidence: 99.76%
-------------------------------------------------------

[Example 3]
Snippet   : "SHOCKING SEC

## ==========================================
## PIPELINE 9: SAVE MODEL TO GOOGLE DRIVE
## ==========================================

In [23]:
from google.colab import drive

print("=" * 60)
print("PIPELINE 9: SAVE MODEL TO GOOGLE DRIVE")
print("=" * 60)

save_directory = "/content/drive/MyDrive/Data Science (FULL)/Project-14"

# Mount Google Drive if not already active
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive storage...")
    drive.mount('/content/drive')

os.makedirs(save_directory, exist_ok=True)

print(f"Exporting model artifacts and tokenizer config to: {save_directory}")
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

saved_files = os.listdir(save_directory)
print("\n--- Export Confirmation Summary ---")
print(f"Successfully saved files in Drive directory: {saved_files}")
print("=" * 60 + "\n")

PIPELINE 9: SAVE MODEL TO GOOGLE DRIVE
Exporting model artifacts and tokenizer config to: /content/drive/MyDrive/Data Science (FULL)/Project-14


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Export Confirmation Summary ---
Successfully saved files in Drive directory: ['Fake News Detection.ipynb', 'config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json']



## ==========================================
## PIPELINE 10: REAL-TIME INFERENCE PIPELINE
## ==========================================

In [25]:
print("=" * 60)
print("PIPELINE 10: REAL-TIME INFERENCE PIPELINE")
print("=" * 60)

model_dir = save_directory
print(f"Loading saved model weights from path: {model_dir}")
inf_tokenizer = AutoTokenizer.from_pretrained(model_dir)
inf_model = AutoModelForSequenceClassification.from_pretrained(model_dir)

inf_model.eval()
sample_article = "BREAKING: Local man discovers an infinite energy source using only household garbage and tape."
inputs = inf_tokenizer(sample_article, return_tensors="pt", truncation=True, max_length=512)

with torch.no_grad():
    outputs = inf_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_label = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][pred_label].item()

label_map = { 0: "Real News ✅",1: "Fake News 🛑"}

print("\n--- Inference Prediction Output ---")
print(f"Input Text Snippet: \"{sample_article}\"")
print(f"Predicted Classification: {label_map[pred_label]}")
print(f"Model Confidence Score: {confidence * 100:.2f}%")
print("=" * 60 + "\n")

PIPELINE 10: REAL-TIME INFERENCE PIPELINE
Loading saved model weights from path: /content/drive/MyDrive/Data Science (FULL)/Project-14


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


--- Inference Prediction Output ---
Input Text Snippet: "BREAKING: Local man discovers an infinite energy source using only household garbage and tape."
Predicted Classification: Fake News 🛑
Model Confidence Score: 99.66%



#THE END